<a href="https://colab.research.google.com/github/Edomario082909/AHHHHR/blob/main/sdxl_v1.0_comfyui_colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import os

# --- 1. CONFIGURAZIONE (METTI IL TUO TOKEN QUI) ---
MY_CIVITAI_TOKEN = "895c00b088efbf2b359254232221ba32"

# --- 2. PREPARAZIONE AMBIENTE ---
!apt -y update -qq && apt -y install -qq aria2
!wget https://github.com/camenduru/gperftools/releases/download/v1.0/libtcmalloc_minimal.so.4 -O /content/libtcmalloc_minimal.so.4
%env LD_PRELOAD=/content/libtcmalloc_minimal.so.4

# --- 3. INSTALLAZIONE COMFYUI E NODI ---
!git clone https://github.com/comfyanonymous/ComfyUI
%cd /content/ComfyUI
!pip install -q -r requirements.txt
%cd /content/ComfyUI/custom_nodes
!git clone https://github.com/ltdrdata/ComfyUI-Manager.git
!git clone https://github.com/kijai/ComfyUI-WanVideo.git
!git clone https://github.com/city96/ComfyUI-GGUF.git
%cd /content/ComfyUI
!pip install -q xformers torchsde omegaconf diffusers accelerate
!pip install -q -r /content/ComfyUI/custom_nodes/ComfyUI-WanVideo/requirements.txt

# --- 4. DOWNLOAD MODELLI BASE WAN 2.1 (T2V e I2V) ---
!aria2c --console-log-level=error -c -x 16 -s 16 -k 1M "https://huggingface.co/Wan-AI/Wan2.1-T2V-1.3B/resolve/main/diffusion_pytorch_model.safetensors" -d /content/ComfyUI/models/checkpoints -o wan2.1_t2v_1.3b.safetensors
!aria2c --console-log-level=error -c -x 16 -s 16 -k 1M "https://huggingface.co/Wan-AI/Wan2.1-I2V-1.3B-480P/resolve/main/diffusion_pytorch_model.safetensors" -d /content/ComfyUI/models/checkpoints -o wan2.1_i2v_1.3b_480p.safetensors
!aria2c --console-log-level=error -c -x 16 -s 16 -k 1M "https://huggingface.co/Wan-AI/Wan2.1-T2V-1.3B/resolve/main/models_vae_diffusion_pytorch_model.safetensors" -d /content/ComfyUI/models/vae -o wan2.1_vae.safetensors

# --- 5. CICLO DI DOWNLOAD AUTOMATICO CIVITAI (TUTTI I TUOI URN) ---
models_to_download = [
    ("2747549", "ltx_lora_fov.safetensors", "loras"),
    ("2754108", "ltx_lora_2.safetensors", "loras"),
    ("2617751", "zimage_lora_1.safetensors", "loras"),
    ("2442439", "zimage_checkpoint.safetensors", "checkpoints"),
    ("2465980", "zimage_lora_2.safetensors", "loras"),
    ("2474435", "zimage_lora_3.safetensors", "loras"),
    ("2083303", "wan_lora_1.safetensors", "loras"),
    ("2200388", "wan_lora_2.safetensors", "loras"),
    ("2475163", "zimage_lora_4.safetensors", "loras"),
    ("2477457", "zimage_lora_5.safetensors", "loras"),
    ("2581135", "zimage_lora_6.safetensors", "loras"),
    ("2273468", "wan_lora_3.safetensors", "loras"),
    ("2441730", "wan_lora_4.safetensors", "loras"),
    ("2553151", "wan_lora_5.safetensors", "loras"),
    ("2460437", "zimage_lora_7.safetensors", "loras"),
    ("2073605", "wan_lora_6.safetensors", "loras"),
    ("2739037", "sdxl_real_checkpoint.safetensors", "checkpoints"),
    ("1811054", "sdxl_lora_1.safetensors", "loras"),
    ("2752410", "ltx_checkpoint.safetensors", "checkpoints"),
    ("2376136", "wan_lora_7.safetensors", "loras"),
    ("2342652", "wan_lora_8.safetensors", "loras"),
    ("2176450", "wan_lora_9.safetensors", "loras"),
    ("2430424", "wan_lora_10.safetensors", "loras"),
    ("2738377", "grok_checkpoint.safetensors", "checkpoints")
]

for version_id, filename, folder in models_to_download:
    dest_path = f"/content/ComfyUI/models/{folder}"
    url = f"https://civitai.com/api/download/models/{version_id}?token={MY_CIVITAI_TOKEN}"
    !aria2c --console-log-level=error -c -x 16 -s 16 -k 1M "{url}" -d {dest_path} -o {filename}

# --- 6. AVVIO SISTEMA CORRETTO ---
!wget https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /content/cloudflared-linux-amd64 && chmod 777 /content/cloudflared-linux-amd64
import atexit, requests, subprocess, time, re
from random import randint
from threading import Timer
from queue import Queue

# Funzione per il tunnel che non blocca l'esecuzione
def start_cloudflared(port, metrics_port):
    subprocess.Popen(['/content/cloudflared-linux-amd64', 'tunnel', '--url', f'http://127.0.0.1:{port}', '--metrics', f'127.0.0.1:{metrics_port}'], stdout=subprocess.DEVNULL, stderr=subprocess.STDOUT)

    tunnel_url = None
    while not tunnel_url:
        time.sleep(2)
        try:
            metrics = requests.get(f'http://127.0.0.1:{metrics_port}/metrics').text
            tunnel_url = re.search("(?P<url>https?:\/\/[^\s]+.trycloudflare.com)", metrics).group("url")
        except:
            pass
    print(f"\n\n🚀 COMFYUI PRONTO! APRI QUESTO LINK:\n{tunnel_url}\n\n")

# Avviamo il cercatore di link in un thread separato
metrics_port = randint(8100, 9000)
threading_timer = Timer(2, start_cloudflared, args=(8188, metrics_port))
threading_timer.start()

# Avviamo ComfyUI (questo bloccherà la cella e la terrà attiva)
!python main.py --dont-print-server --preview-method auto